# ML06：学习 conditional diffusion 与 flow

在相同合成双峰 action distribution 上训练两个小网络，再从噪声采样；不使用 oracle endpoint 作为推理输入。

**运行方式**：上传本 `.ipynb` 到 Google Colab，连接 Python runtime，从上到下运行。
默认 CPU 即可；本 notebook 自带代码和合成数据，不需要克隆仓库、上传论文、挂载 Drive 或配置本地环境。
所有输出留空，首次运行后才产生实际结果；不要把材料准备状态当作已复现实验。

如果所选云端 runtime 缺少包，可在 Colab 新单元执行 `%pip install numpy matplotlib pandas scikit-learn torch`，
然后重启 runtime。不要为了本课程在本机安装依赖。


## 1. 环境与随机性
记录版本和 seed。换 seed 时从本单元重新运行；图中用英文标签避免依赖额外中文字体。


In [ ]:
import json
import math
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
print('Python:', platform.python_version(), '| NumPy:', np.__version__, '| seed:', SEED)
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(SEED)
torch.set_num_threads(2)
print('torch:', torch.__version__, '| teaching device: CPU')


## 2. 同一数据、同一 MLP 结构
每条样本独立，train/validation/test 隔离。hidden branch 不输入网络；不把采样模式距离叫 robot success。


In [ ]:
import copy
from time import perf_counter
gen = torch.Generator().manual_seed(SEED+30)
observations = 2*torch.rand(3600, 1, generator=gen)-1
branches = 2*torch.randint(0, 2, (3600, 1), generator=gen)-1
actions = .5*observations + branches + .05*torch.randn(3600, 1, generator=gen)
train_obs, train_act = observations[:2400], actions[:2400]
val_obs, val_act = observations[2400:3000], actions[2400:3000]
test_obs, test_act = observations[3000:], actions[3000:]
class ConditionalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 1))
    def forward(self, state, obs, time):
        return self.net(torch.cat([state, obs, time], dim=-1))
initial = ConditionalNet()
flow, diffusion = copy.deepcopy(initial), copy.deepcopy(initial)
TRAIN_STEPS, BATCH_SIZE = 800, 128


## 3. Flow：noise→data，target=a−epsilon
采样时网络只读当前 x、observation 和 tau。固定训练预算，不依据 test 选择 steps。


In [ ]:
optimizer = torch.optim.Adam(flow.parameters(), lr=.002)
flow_trace = []
for step in range(TRAIN_STEPS):
    ids = torch.randint(len(train_obs), (BATCH_SIZE,))
    obs, endpoint = train_obs[ids], train_act[ids]
    noise = torch.randn_like(endpoint); tau = torch.rand(BATCH_SIZE, 1)
    state = (1-tau)*noise + tau*endpoint
    target = endpoint-noise
    objective = F.mse_loss(flow(state, obs, tau), target)
    optimizer.zero_grad(set_to_none=True); objective.backward(); optimizer.step()
    if step % 20 == 0:
        flow_trace.append((step, float(objective.detach())))


## 4. DDPM：预测 epsilon
固定 48-step schedule。训练随机采 k；采样再走完整 reverse chain。两个 loss 目标不同，不横比数值大小。


In [ ]:
DIFFUSION_STEPS = 48
beta = torch.linspace(.001, .2, DIFFUSION_STEPS)
alpha = 1-beta; alpha_bar = torch.cumprod(alpha, dim=0)
alpha_bar_previous = torch.cat([torch.ones(1), alpha_bar[:-1]])
posterior_variance = beta*(1-alpha_bar_previous)/(1-alpha_bar)
optimizer = torch.optim.Adam(diffusion.parameters(), lr=.002)
diffusion_trace = []
for step in range(TRAIN_STEPS):
    ids = torch.randint(len(train_obs), (BATCH_SIZE,))
    obs, endpoint = train_obs[ids], train_act[ids]
    k = torch.randint(DIFFUSION_STEPS, (BATCH_SIZE,))
    noise = torch.randn_like(endpoint)
    cumulative = alpha_bar[k, None]
    state = cumulative.sqrt()*endpoint + (1-cumulative).sqrt()*noise
    time = k[:, None].float()/(DIFFUSION_STEPS-1)
    objective = F.mse_loss(diffusion(state, obs, time), noise)
    optimizer.zero_grad(set_to_none=True); objective.backward(); optimizer.step()
    if step % 20 == 0:
        diffusion_trace.append((step, float(objective.detach())))
print('Final forward alpha_bar:', float(alpha_bar[-1]))


## 5. 两个 inference sampler：不接受真实 endpoint
Flow 使用 Euler；DDPM 使用 epsilon prediction 对应的 reverse mean 与 posterior variance。输出每个 observation 的一个 sample。


In [ ]:
@torch.no_grad()
def sample_flow(model, obs, steps=32):
    model.eval(); state = torch.randn(len(obs), 1)
    for index in range(steps):
        tau = torch.full((len(obs), 1), index/steps)
        state = state + model(state, obs, tau)/steps
    return state
@torch.no_grad()
def sample_diffusion(model, obs):
    model.eval(); state = torch.randn(len(obs), 1)
    for k in reversed(range(DIFFUSION_STEPS)):
        time = torch.full((len(obs), 1), k/(DIFFUSION_STEPS-1))
        predicted_noise = model(state, obs, time)
        mean = (state-beta[k]/torch.sqrt(1-alpha_bar[k])*predicted_noise)/torch.sqrt(alpha[k])
        state = mean if k == 0 else mean+torch.sqrt(posterior_variance[k])*torch.randn_like(state)
    return state


## 6. 固定配置的 held-out 采样指标与 o=0 分布
记录采样耗时，不把 CPU wall time 推广成真实 VLA latency。曲线不保证一次训练就覆盖所有模式。


In [ ]:
metrics = {}; sample_sets = {}
samplers = {'flow': lambda obs: sample_flow(flow, obs, steps=32),
            'diffusion': lambda obs: sample_diffusion(diffusion, obs)}
for name, sampler in samplers.items():
    torch.manual_seed(SEED+100)
    begin = perf_counter(); samples = sampler(test_obs); elapsed = perf_counter()-begin
    centered = samples-.5*test_obs
    distance = torch.minimum((centered-1).abs(), (centered+1).abs())
    metrics[name] = {'test_sample_mode_compatibility': float((distance<.15).float().mean()),
                     'test_positive_mode_fraction': float((centered>0).float().mean()),
                     'sampling_seconds_cpu': elapsed, 'sample_count': len(test_obs)}
    sample_sets[name] = sampler(torch.zeros(2000, 1)).numpy().ravel()
results = {'seed': SEED, 'train_steps_per_model': TRAIN_STEPS, 'metrics': metrics,
           'mode_distance_threshold': .15, 'flow_sampling_steps': 32,
           'diffusion_sampling_steps': DIFFUSION_STEPS,
           'scope': 'Learned 1D conditional synthetic distributions. No robot rollout or paper reproduction.'}
display(pd.DataFrame(metrics).T)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for axis, (name, samples) in zip(axes, sample_sets.items()):
    axis.hist(samples, bins=60, range=(-2, 2), density=True, alpha=.7)
    axis.axvline(-1, linestyle='--', color='black'); axis.axvline(1, linestyle='--', color='black')
    axis.set(title=name, xlabel='sample at o=0', ylabel='density')
plt.tight_layout(); plt.show()


## 面试追问与实验记录
1. 手推 flow target，解释为什么 sampler 不读 expert endpoint。
2. Diffusion 训练随机抽一步，为什么采样仍是多步？
3. 在 validation 上尝试不同 Euler steps，固定配置后再查看 test。
4. 为什么 training MSE、mode coverage 与 closed-loop performance 是不同指标？

记录 Question / Data / Config / Metric / Observation / Explanation / Limitation。
改变实验设置时不要反复用 test 选择答案；保存 notebook 后再下载 `.ipynb` 保留输出。


## 可选：导出指标
默认关闭。需要时打开开关；文件只写到当前 **Colab runtime** 的 `outputs/`。runtime 临时文件不等于已永久保存，可使用 Colab 文件面板下载。


In [ ]:
EXPORT_RESULTS = False
if EXPORT_RESULTS:
    destination = Path('outputs')
    destination.mkdir(exist_ok=True)
    (destination / 'metrics.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    print('Saved outputs/metrics.json in the Colab runtime; download it from the Files panel.')
